In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

In [3]:
file_path = "movies.csv"
movies_df = pd.read_csv(file_path)
file_path2 = "ratings_sample.csv"
rating_df = pd.read_csv(file_path2)
movies_df.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [5]:
movies_df['year'] = movies_df.title.str.extract("(\(\d\d\d\d\))", expand=False)
movies_df['year'] = movies_df['year'].str.extract("(\d\d\d\d)", expand=False)
movies_df['title'] = movies_df['title'].str.replace('(\(\d\d\d\d\))', '', regex=True)
movies_df['title'] = movies_df['title'].apply(lambda x: x.strip())
movies_df.head()

,movieId,title,genres,year
0,1,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,1995
1,2,Jumanji,Adventure|Children|Fantasy,1995
2,3,Grumpier Old Men,Comedy|Romance,1995
3,4,Waiting to Exhale,Comedy|Drama|Romance,1995
4,5,Father of the Bride Part II,Comedy,1995


In [7]:
movies_df = movies_df.drop('genres', axis=1)
movies_df.head()

,movieId,title,year
0,1,Toy Story,1995
1,2,Jumanji,1995
2,3,Grumpier Old Men,1995
3,4,Waiting to Exhale,1995
4,5,Father of the Bride Part II,1995


In [9]:
rating_df.head()

,userId,movieId,rating,timestamp
0,1,169,2.5,1204927694
1,1,2471,3.0,1204927438
2,1,48516,5.0,1204927435
3,2,2571,3.5,1436165433
4,2,109487,4.0,1436165496


In [11]:
rating_df = rating_df.drop('timestamp', axis=1)
rating_df.head()

,userId,movieId,rating
0,1,169,2.5
1,1,2471,3.0
2,1,48516,5.0
3,2,2571,3.5
4,2,109487,4.0


In [90]:
if 'userId' in rating_df.columns and rating_df['userId'].dtype != 'int64':
    rating_df['userId'] = pd.to_numeric(rating_df['userId'], errors='coerce')
    rating_df.dropna(subset=['userId'], inplace=True)
    rating_df['userId'] = rating_df['userId'].astype('int64')

if 'movieId' in rating_df.columns and rating_df['movieId'].dtype != 'int64':
    rating_df['movieId'] = pd.to_numeric(rating_df['movieId'], errors='coerce')
    rating_df['movieId'] = rating_df['movieId'].astype('int64')

if 'movieId' in movies_df.columns and movies_df['movieId'].dtype != 'int64':
    movies_df['movieId'] = pd.to_numeric(movies_df['movieId'], errors='coerce')
    movies_df['movieId'] = movies_df['movieId'].astype('int64')

## Collaborative Filtering

In [93]:
user_Id = 2
userInput = rating_df[rating_df['userId'] == user_Id]
inputMovies = pd.merge(userInput, movies_df, on='movieId')
inputMovies

,userId,movieId,rating,title,year
0,2,2571,3.5,"Matrix, The",1999
1,2,109487,4.0,Interstellar,2014
2,2,112552,5.0,Whiplash,2014
3,2,112556,4.0,Gone Girl,2014


In [95]:
inputMovies = inputMovies.drop('year', axis=1)
inputMovies.head()

,userId,movieId,rating,title
0,2,2571,3.5,"Matrix, The"
1,2,109487,4.0,Interstellar
2,2,112552,5.0,Whiplash
3,2,112556,4.0,Gone Girl


### the users who has seen the same movies

In [98]:
userSubset = rating_df[rating_df['movieId'].isin(inputMovies['movieId'].tolist())]
userSubset.head(10)

,userId,movieId,rating
3,2,2571,3.5
4,2,109487,4.0
5,2,112552,5.0
6,2,112556,4.0
105,4,2571,4.0
195,5,2571,2.5
202,5,112552,4.5
203,5,112556,4.5
252,7,109487,5.0
356,11,2571,4.5


In [100]:
userSubset.shape

(14689, 3)

In [102]:
userSubsetGroup = userSubset.groupby(['userId'])

In [104]:
userSubsetGroup.get_group(1130)

,userId,movieId,rating
104530,1130,2571,2.0


In [110]:
userSubsetGroup = sorted(userSubsetGroup, key=lambda x: len(x[1]), reverse=True)
userSubsetGroup[0:3]

[((2,),
     userId  movieId  rating
  3       2     2571     3.5
  4       2   109487     4.0
  5       2   112552     5.0
  6       2   112556     4.0),
 ((115,),
         userId  movieId  rating
  11171     115     2571     5.0
  11222     115   109487     4.0
  11224     115   112552     3.5
  11225     115   112556     5.0),
 ((128,),
         userId  movieId  rating
  12405     128     2571     3.5
  12410     128   109487     5.0
  12412     128   112552     4.0
  12413     128   112556     5.0)]

## Similarity of users to input user

In [125]:
from scipy.stats import pearsonr
pearsoncorrelationdict = {}
for user_tuple, user_rating in userSubsetGroup :
    user_name = user_tuple[0]
    
    commonMovies = inputMovies[inputMovies['movieId'].isin(user_rating['movieId'])]
    
    userRatingSorted = user_rating.sort_values(by='movieId')
    commonMoviesSorted = commonMovies.sort_values(by='movieId')

    if len(commonMoviesSorted) == 0 :
        pearsoncorrelationdict[user_name] = 0
        continue

    inputRating = commonMoviesSorted['rating'].tolist()
    userRatingList = userRatingSorted[userRatingSorted['movieId'].isin(commonMoviesSorted['movieId'])]['rating'].tolist()

    if len(inputRating) > 1:
        corr, _ = pearsonr(inputRating, userRatingList)
        pearsoncorrelationdict[user_name] = corr
    else:
       pearsoncorrelationdict[user_name] = 0


In [127]:
pearsoncorrelationdict

{2: 0.9999999999999998,
 115: -0.838870492807861,
 128: 0.04415107856883482,
 176: 0.13245323570650436,
 237: -0.05263157894736839,
 383: -0.13245323570650436,
 409: 0.7894736842105261,
 474: -0.32444284226152503,
 598: 0.5129891760425771,
 632: 0.5739640213948523,
 815: 0.0,
 820: 0.3823595564509362,
 1011: -0.8992288030258968,
 1040: -0.9233805168766386,
 1209: -0.64888568452305,
 1337: -0.760885910252682,
 1502: 0.20751433915982237,
 1660: -0.973328526784575,
 1669: nan,
 1848: -0.3077935056255462,
 2149: 0.838870492807861,
 2169: 0.7492686492653552,
 2253: 0.6882472016116852,
 2274: 0.17492785713532985,
 2452: -0.20751433915982237,
 2604: -0.029867384554474852,
 2726: -0.13245323570650436,
 2788: -0.4842001247062522,
 2814: 0.32444284226152503,
 2824: -0.13245323570650436,
 3119: -0.6622661785325219,
 3130: -0.7059732072369207,
 3309: 0.6882472016116852,
 3508: -0.6882472016116852,
 3577: 0.13245323570650436,
 3699: 0.64888568452305,
 3734: -0.512989176042577,
 3768: -0.66226617853

In [129]:
pearsonDF = pd.DataFrame.from_dict(pearsoncorrelationdict, orient='index')
pearsonDF.columns = ['similarityIndex']
pearsonDF['userId'] = pearsonDF.index 
pearsonDF.index = range(len(pearsonDF))
pearsonDF.head()

,similarityIndex,userId
0,1.000000,2
1,-0.838870,115
2,0.044151,128
3,0.132453,176
4,-0.052632,237


In [131]:
topUser = pearsonDF.sort_values(by='similarityIndex', ascending=False)[0:50]
topUser.head()

,similarityIndex,userId
1755,1.0,35644
1666,1.0,31593
1213,1.0,13007
1777,1.0,36679
1215,1.0,13107


In [133]:
print(topUser['userId'].dtype)
print(rating_df['userId'].dtype)

int64
int64


In [135]:
topUser['userId'] = topUser['userId'].apply(lambda x: x[0] if isinstance(x, (list, np.ndarray)) else x)
topUser['userId'] = pd.to_numeric(topUser['userId'], errors='coerce')
topUser.dropna(subset=['userId'], inplace=True)
topUser['userId'] = topUser['userId'].astype('int64')
print(topUser['userId'].dtype)

int64


In [137]:
common_ids = set(topUser['userId']).intersection(set(rating_df['userId']))
print(len(common_ids))

50


In [145]:
topUserRating = topUser.merge(rating_df, left_on='userId', right_on='userId', how='inner')
topUserRating.head()

,similarityIndex,userId,movieId,rating
0,1.0,35644,1,3.5
1,1.0,35644,296,5.0
2,1.0,35644,318,4.5
3,1.0,35644,364,5.0
4,1.0,35644,784,3.0


In [151]:
topUserRating['weightedRating'] = topUserRating['similarityIndex'] * topUserRating['rating']
topUserRating.head()

,similarityIndex,userId,movieId,rating,weightedRating
0,1.0,35644,1,3.5,3.5
1,1.0,35644,296,5.0,5.0
2,1.0,35644,318,4.5,4.5
3,1.0,35644,364,5.0,5.0
4,1.0,35644,784,3.0,3.0


In [157]:
temptopUserRating = topUserRating.groupby('movieId').sum()[['similarityIndex','weightedRating']]
temptopUserRating.columns = ['sum_similarityIndex','sum_weightedRating']
temptopUserRating.head()

,sum_similarityIndex,sum_weightedRating
movieId,,
1,23.0,82.5
2,5.0,13.5
5,1.0,2.5
6,3.0,12.5
10,4.0,12.5


In [161]:
recommendation_df = pd.DataFrame()
recommendation_df['weighted average recommendation score'] = temptopUserRating['sum_weightedRating']/temptopUserRating['sum_similarityIndex']
recommendation_df['movieId'] = temptopUserRating.index
recommendation_df.head()

,weighted average recommendation score,movieId
movieId,,
1,3.586957,1
2,2.700000,2
5,2.500000,5
6,4.166667,6
10,3.125000,10


In [163]:
recommendation_df = recommendation_df.sort_values(by='weighted average recommendation score', ascending=False)
recommendation_df.head(10)

,weighted average recommendation score,movieId
movieId,,
151315,5.0,151315
49772,5.0,49772
7346,5.0,7346
106785,5.0,106785
82152,5.0,82152
82378,5.0,82378
68848,5.0,68848
1111,5.0,1111
4467,5.0,4467


In [165]:
movies_df.loc[movies_df['movieId'].isin(recommendation_df.head(10)['movieId'].tolist())]

,movieId,title,year
1088,1111,Microcosmos (Microcosmos: Le peuple de l'herbe),1996
4373,4467,"Adventures of Baron Munchausen, The",1988
7235,7346,"Girl Next Door, The",2004
11519,49772,"Painted Veil, The",2006
13669,68159,State of Play,2009
13764,68848,"Brothers Bloom, The",2008
16300,82152,Beastly,2011
16356,82378,All Good Things,2010
22293,106785,Homefront,2013
34117,151315,Ride Along 2,2016
